# 🖼️ Computational Imaging – Group W
## Motion Deblur & Denoising on FFHQ 256 × 256

**Project spec recap**

| # | Family | Algorithm |
|---|--------|-----------|
| 1 | Variational | Total *p*-Variation (TpV, *p* ∈ (0.1 – 0.5)) via ADMM |
| 2 | End-to-end | U-Net (supervised) |
| 3 | Generative | DDPM + DPS (Diffusion Posterior Sampling) |
| 4 | Hybrid | Plug-and-Play HQS (PnP-HQS) |

Each section is self-contained. Every `# TODO` marks a student decision to fill in.

> **How to use:** Runtime → Run all, then work through the TODOs top-to-bottom.


## 0 · Setup & Installations
Install all required libraries that are not available by default in Colab.

In [3]:
# Install extra packages (run once per Colab session)
!pip install -q datasets tqdm scikit-image Pillow torchvision matplotlib
# datasets  → HuggingFace FFHQ loader
# scikit-image → PSNR / SSIM evaluation metrics
print("All packages installed ✓")


All packages installed ✓


In [ ]:
import math, glob, os, time, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

warnings.filterwarnings('ignore')

# ── Device selection (same helper used in all course files) ──────────────────
def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    try:
        if torch.backends.mps.is_available():
            return 'mps'
    except AttributeError:
        pass
    return 'cpu'

device = get_device()
print(f"Using device: {device}")

# ── Global folders ────────────────────────────────────────────────────────────
WEIGHTS_DIR = Path('./content/weights')
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR    = Path('./content/ffhq256')
DATA_DIR.mkdir(parents=True, exist_ok=True)


Using device: mps


## 1 · Dataset — FFHQ 256 × 256

We load the **FFHQ-256** dataset from HuggingFace (`bitmind/ffhq-256`).

The preprocessing pipeline (which you must justify in the report) is:
1. Convert RGB → **grayscale** (the task is image restoration, colour is not required)
2. Resize to exactly **256 × 256**
3. Normalise pixel values to **[0, 1]** (float32)
4. Split into **train / validation / test** subsets

> **TODO 1-A** – Choose the train/val/test split ratio and justify it in your report.

> **TODO 1-B** – Decide how many total images to use (the full 70 k is fine but slow; a subset is acceptable if you justify it).


In [ ]:
from datasets import load_dataset

# ── TODO 1-A: Set split sizes ──────────────────────────────────────────────
N_TRAIN = 5000   # number of images for training
N_VAL   = 500    # number of images for validation
N_TEST  = 200    # number of images held out for final evaluation
# ──────────────────────────────────────────────────────────────────────────────

print("Streaming FFHQ-256 from HuggingFace (first run may take a few minutes)…")
hf_dataset = load_dataset(
    "bitmind/ffhq-256",
    split="train",          # FFHQ has only one split on HuggingFace
    streaming=False,
)
print(f"Total images available: {len(hf_dataset)}")

# ── Reproducible shuffle then split ─────────────────────────────────────────
import random
random.seed(42)
indices = list(range(len(hf_dataset)))
random.shuffle(indices)

train_indices = indices[:N_TRAIN]
val_indices   = indices[N_TRAIN : N_TRAIN + N_VAL]
test_indices  = indices[N_TRAIN + N_VAL : N_TRAIN + N_VAL + N_TEST]

print(f"Split → train: {len(train_indices)} | val: {len(val_indices)} | test: {len(test_indices)}")


Streaming FFHQ-256 from HuggingFace (first run may take a few minutes)…


Generating train split: 100%|██████████| 70000/70000 [00:12<00:00, 5780.26 examples/s]


Total images available: 70000
Split → train: 5000 | val: 500 | test: 200


In [11]:
# ── FFHQ Dataset class ────────────────────────────────────────────────────────
# Adapted from the MayoDataset pattern used throughout the course files.
# Key differences: source is HuggingFace (PIL images), input is RGB → grayscale.

class FFHQDataset(Dataset):
    """
    Wraps a subset of the HuggingFace FFHQ-256 dataset.

    Parameters
    ----------
    hf_dataset : HuggingFace Dataset object
    indices    : list[int] – indices to include in this split
    img_size   : int – output spatial resolution (default 256)
    """
    def __init__(self, hf_dataset, indices, img_size=256):
        super().__init__()
        self.hf_dataset = hf_dataset
        self.indices    = indices
        self.transform  = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),   # RGB → L
            transforms.Resize((img_size, img_size)),        # ensure 256×256
            transforms.ToTensor(),                          # [0,255] → [0,1] float32
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        pil_img  = self.hf_dataset[real_idx]['image']      # PIL RGB image
        return self.transform(pil_img)                      # (1, 256, 256)


train_dataset = FFHQDataset(hf_dataset, train_indices)
val_dataset   = FFHQDataset(hf_dataset, val_indices)
test_dataset  = FFHQDataset(hf_dataset, test_indices)

# ── TODO 1-B: Adjust batch size according to GPU memory ───────────────────
BATCH_TRAIN = 8
BATCH_EVAL  = 4
# ──────────────────────────────────────────────────────────────────────────────

train_loader = DataLoader(train_dataset, batch_size=BATCH_TRAIN, shuffle=True,
                          num_workers=2, pin_memory=(device == 'cuda'))
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_EVAL,  shuffle=False,
                          num_workers=2, pin_memory=(device == 'cuda'))
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_EVAL,  shuffle=False,
                          num_workers=2, pin_memory=(device == 'cuda'))

print(f"Loaders ready — train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)}")


Loaders ready — train: 5000 | val: 500 | test: 200


In [12]:
# ── Quick sanity-check: visualise a mini-batch ──────────────────────────────
sample_batch = next(iter(train_loader))           # (B, 1, 256, 256)
print(f"Batch shape : {tuple(sample_batch.shape)}")
print(f"Dtype       : {sample_batch.dtype}")
print(f"Value range : [{sample_batch.min():.3f}, {sample_batch.max():.3f}]")

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, img in zip(axes, sample_batch[:6]):
    ax.imshow(img.squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle('Sample FFHQ grayscale images (256×256, normalised [0,1])')
plt.tight_layout()
plt.show()


Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=96, pipe_handle=133)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=96, pipe_handle=126)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: module '__main__' has no attribute 'FFHQDataset'
  File "/opt/hom

RuntimeError: DataLoader worker (pid(s) 6344, 6345) exited unexpectedly

## 2 · Degradation Model

The forward (degradation) operator is:

$$y^\delta = K x + \eta, \quad \|\eta\| = \delta \|y\|$$

where **K** is a motion-blur convolution with:
- `kernel_size = 9` (fixed by the spec)
- `motion_angle` → **TODO 2-A** (your choice, must be justified)

and $\delta$ ∈ {0.005, 0.01, 0.05, 0.1} (all four tested in the evaluation).

The Gaussian noise is generated using the **relative** noise model from the course
(`‖e‖ / ‖y‖ = noise_level`), which is the exact same function used in all course scripts.


In [ ]:
# ── Gaussian noise (identical to the IPPy / course implementation) ───────────
def gaussian_noise(y: torch.Tensor, noise_level: float) -> torch.Tensor:
    """
    Returns additive Gaussian noise e such that ‖e‖ / ‖y‖ = noise_level.
    This is the relative noise model used throughout the course.
    """
    e = torch.randn_like(y)
    return e / torch.norm(e) * torch.norm(y) * noise_level


# ── Motion-blur kernel (pure PyTorch, no IPPy dependency) ────────────────────
def make_motion_kernel(kernel_size: int, angle_deg: float) -> torch.Tensor:
    """
    Build a 1-D motion-blur PSF of size (kernel_size × kernel_size) at angle_deg.
    Returns a float32 tensor normalised to sum = 1.
    """
    k = kernel_size
    kernel = torch.zeros(k, k)
    cx, cy = k // 2, k // 2
    angle_rad = math.radians(angle_deg)
    for i in range(k):
        t  = i - k // 2
        x  = round(cx + t * math.cos(angle_rad))
        y  = round(cy + t * math.sin(angle_rad))
        if 0 <= x < k and 0 <= y < k:
            kernel[x, y] = 1.0
    kernel /= kernel.sum()
    return kernel   # (k, k)


class MotionBlurOperator:
    """
    Differentiable motion-blur convolution applied batch-wise.
    Mirrors the IPPy Blurring operator interface used in course scripts.

    Parameters
    ----------
    kernel_size  : int (9 – fixed by spec)
    motion_angle : float in degrees
    device       : torch device
    """
    def __init__(self, kernel_size: int, motion_angle: float, device):
        self.kernel_size  = kernel_size
        self.motion_angle = motion_angle
        self.device       = device
        raw = make_motion_kernel(kernel_size, motion_angle)          # (k, k)
        # Store as (1, 1, k, k) for F.conv2d / F.conv_transpose2d
        self.kernel = raw.unsqueeze(0).unsqueeze(0).to(device)       # (1,1,k,k)

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        """Apply blur: x (B,1,H,W) → y (B,1,H,W)."""
        B, C, H, W = x.shape
        pad = self.kernel_size // 2
        return F.conv2d(x, self.kernel, padding=pad)

    def T(self, y: torch.Tensor) -> torch.Tensor:
        """Adjoint (transpose) convolution."""
        pad = self.kernel_size // 2
        return F.conv_transpose2d(y, self.kernel, padding=pad)


# ── TODO 2-A: Choose the motion angle (document your choice in the report) ─
MOTION_ANGLE = 45       # degrees – e.g. 20, 30, 45, 60 …
# ──────────────────────────────────────────────────────────────────────────────

K = MotionBlurOperator(kernel_size=9, motion_angle=MOTION_ANGLE, device=device)
NOISE_LEVELS = [0.005, 0.01, 0.05, 0.1]     # all four levels from the spec

print(f"Motion-blur operator ready — kernel 9×9, angle {MOTION_ANGLE}°")
print(f"Noise levels to test: {NOISE_LEVELS}")


In [ ]:
# ── Visualise degradation at all four noise levels ───────────────────────────
x_sample = test_dataset[0].unsqueeze(0).to(device)     # (1,1,256,256)

with torch.no_grad():
    y_clean = K(x_sample)

fig, axes = plt.subplots(2, len(NOISE_LEVELS) + 1, figsize=(16, 7))
axes[0, 0].imshow(x_sample.cpu().squeeze(), cmap='gray'); axes[0, 0].set_title('Ground truth'); axes[0, 0].axis('off')
axes[1, 0].imshow(y_clean.cpu().squeeze(),  cmap='gray'); axes[1, 0].set_title('Blurred (no noise)'); axes[1, 0].axis('off')

for col, nl in enumerate(NOISE_LEVELS, start=1):
    with torch.no_grad():
        y_noisy = y_clean + gaussian_noise(y_clean, nl)
    axes[0, col].imshow(y_noisy.cpu().squeeze(), cmap='gray')
    axes[0, col].set_title(f'δ = {nl}'); axes[0, col].axis('off')
    diff = (x_sample - y_noisy).abs().cpu().squeeze()
    axes[1, col].imshow(diff, cmap='hot'); axes[1, col].set_title('|residual|'); axes[1, col].axis('off')

plt.suptitle('Degradation pipeline at each noise level')
plt.tight_layout()
plt.show()


## 3 · Shared Evaluation Utilities

These functions are used by **all four methods** to ensure a fair comparison on identical degraded inputs.


In [ ]:
# ── Metric helpers ────────────────────────────────────────────────────────────
def compute_metrics(x_true: torch.Tensor, x_pred: torch.Tensor):
    """
    Compute PSNR and SSIM between two (B,1,H,W) tensors in [0,1].
    Returns averaged scalars over the batch.
    """
    x_true = x_true.detach().cpu().clamp(0, 1)
    x_pred = x_pred.detach().cpu().clamp(0, 1)
    psnrs, ssims = [], []
    for xt, xp in zip(x_true, x_pred):
        xt_np = xt.squeeze().numpy()
        xp_np = xp.squeeze().numpy()
        psnrs.append(peak_signal_noise_ratio(xt_np, xp_np, data_range=1.0))
        ssims.append(structural_similarity(xt_np, xp_np,   data_range=1.0))
    return float(np.mean(psnrs)), float(np.mean(ssims))


def show_comparison(x_true, y_delta, reconstructions: dict, noise_level):
    """
    Side-by-side comparison of all reconstructions for one test image.
    reconstructions = {'Method name': tensor (1,1,H,W), ...}
    """
    n   = len(reconstructions)
    fig, axes = plt.subplots(1, n + 2, figsize=(4 * (n + 2), 4))
    axes[0].imshow(x_true.cpu().squeeze(), cmap='gray'); axes[0].set_title('Ground truth'); axes[0].axis('off')
    axes[1].imshow(y_delta.cpu().squeeze(), cmap='gray'); axes[1].set_title(f'Degraded\n(δ={noise_level})'); axes[1].axis('off')
    for ax, (name, xr) in zip(axes[2:], reconstructions.items()):
        psnr, ssim = compute_metrics(x_true, xr)
        ax.imshow(xr.detach().cpu().squeeze(), cmap='gray')
        ax.set_title(f'{name}\nPSNR:{psnr:.2f} SSIM:{ssim:.3f}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()


# ── Reference: degraded inputs that every method will receive ────────────────
# We fix one test image and one noise level for quick visual debugging.
DEMO_NL = 0.01      # noise level used for quick visual checks

x_demo = test_dataset[0].unsqueeze(0).to(device)
with torch.no_grad():
    y_demo_clean = K(x_demo)
    y_demo       = y_demo_clean + gaussian_noise(y_demo_clean, DEMO_NL)

print("Shared demo degraded image ready (shape:", tuple(y_demo.shape), ")")


## 4 · Method 1 — Variational: Total *p*-Variation (TpV)

### Theory
We solve the regularised inverse problem:

$$\hat{x} = \arg\min_x \frac{1}{2}\|Kx - y^\delta\|_2^2 + \lambda \, \text{TV}_p(x)$$

where the **Total *p*-Variation** semi-norm is:

$$\text{TV}_p(x) = \sum_{i,j} \left( |\nabla_h x_{i,j}|^2 + |\nabla_v x_{i,j}|^2 \right)^{p/2}, \quad p \in (0, 1)$$

For $p < 1$ this is **non-convex** and promotes sparser (sharper) edges than standard TV ($p=1$).

### Algorithm — ADMM
We use the **Alternating Direction Method of Multipliers (ADMM)** splitting:

$$\min_{x,z} \frac{1}{2}\|Kx - y\|^2 + \lambda\|z\|_p^p, \quad \text{s.t.} \; Dx = z$$

which alternates:
1. **x-update**: linear solve (data fidelity + quadratic penalty)
2. **z-update**: proximal of $\|\cdot\|_p^p$ (element-wise shrinkage)
3. **u-update**: dual variable ascent

> **TODO 4-A** – Set the regularisation parameter `lam` (try 1e-3 to 5e-2).

> **TODO 4-B** – Set the exponent `p` (must be in (0.1, 0.5) per spec).

> **TODO 4-C** – Set the ADMM penalty `rho` and number of iterations `n_iter`.


In [ ]:
# ── Finite-difference gradient operators ──────────────────────────────────────
def grad_h(x):
    """Horizontal finite difference (right - centre), zero-padded."""
    return F.pad(x[:, :, :, 1:] - x[:, :, :, :-1], (0, 1, 0, 0))

def grad_v(x):
    """Vertical finite difference (down - centre), zero-padded."""
    return F.pad(x[:, :, 1:, :] - x[:, :, :-1, :], (0, 0, 0, 1))

def div_h(p):
    """Adjoint of grad_h (divergence)."""
    return F.pad(p[:, :, :, :-1] - p[:, :, :, 1:], (1, 0, 0, 0))

def div_v(p):
    """Adjoint of grad_v (divergence)."""
    return F.pad(p[:, :, :-1, :] - p[:, :, 1:, :], (0, 0, 1, 0))

def D(x):
    """Stack [grad_h, grad_v] → shape (B, 2, H, W)."""
    return torch.cat([grad_h(x), grad_v(x)], dim=1)

def Dt(z):
    """Adjoint D^T of D: z (B,2,H,W) → (B,1,H,W)."""
    return -div_h(z[:, :1]) - div_v(z[:, 1:])


In [ ]:
# ── Proximal operator for ‖·‖_p^p (generalised shrinkage for p<1) ────────────
def prox_lp(z: torch.Tensor, lam: float, p: float, eps: float = 1e-8) -> torch.Tensor:
    """
    Element-wise proximal operator for f(z) = lam * |z|^p, p in (0,1).

    Uses the iterative half-quadratic minimisation closed-form (one Newton step):
        prox_{lam * |·|^p}(v) ≈ sign(v) * max(|v| - lam*p*|u|^{p-1}/2, 0)
    evaluated at the current iterate u=v (sufficient for small lam).

    Reference: Candes et al. 2008, Chartrand 2007.
    """
    absz = z.abs().clamp(min=eps)
    thresh = lam * p * absz.pow(p - 1)     # soft-threshold value
    return z.sign() * F.relu(absz - thresh)


# ── TpV-ADMM solver ──────────────────────────────────────────────────────────
def tpv_admm(y: torch.Tensor,
             K,
             lam: float,
             p: float,
             rho: float,
             n_iter: int,
             cg_iter: int = 20) -> torch.Tensor:
    """
    Solve  min_x  0.5*‖Kx - y‖² + λ*TVp(x)  via ADMM.

    Parameters
    ----------
    y       : degraded observation  (1, 1, H, W)
    K       : MotionBlurOperator
    lam     : regularisation weight
    p       : TV exponent in (0, 1)
    rho     : ADMM penalty parameter
    n_iter  : outer ADMM iterations
    cg_iter : conjugate-gradient steps for the x-sub-problem

    Returns
    -------
    x : reconstructed image (1, 1, H, W)
    """
    # Initialise primal, auxiliary, dual variables
    x = y.clone()
    z = D(x)                        # (1, 2, H, W)
    u = torch.zeros_like(z)

    for it in range(n_iter):
        # ── x-update: (K^T K + rho D^T D) x = K^T y + rho D^T(z - u) ──────
        # Solved with Conjugate Gradient
        rhs = K.T(y) + rho * Dt(z - u)

        def Ax(v):
            return K.T(K(v)) + rho * Dt(D(v))

        # CG
        r = rhs - Ax(x)
        d = r.clone()
        rs_old = (r * r).sum()
        for _ in range(cg_iter):
            Ad   = Ax(d)
            alpha = rs_old / ((d * Ad).sum() + 1e-12)
            x    = x + alpha * d
            r    = r - alpha * Ad
            rs_new = (r * r).sum()
            if rs_new.sqrt() < 1e-6:
                break
            d      = r + (rs_new / (rs_old + 1e-12)) * d
            rs_old = rs_new

        # ── z-update: prox of lam/rho * ‖·‖_p^p ─────────────────────────────
        z = prox_lp(D(x) + u, lam / rho, p)

        # ── u-update (dual ascent) ────────────────────────────────────────────
        u = u + D(x) - z

    return x.clamp(0.0, 1.0)


In [ ]:
# ── TODO 4-A: Regularisation parameter ──────────────────────────────────────
TPV_LAM = 0.02       # try range [1e-3, 5e-2]
# ── TODO 4-B: TpV exponent ───────────────────────────────────────────────────
TPV_P   = 0.3        # must be in (0.1, 0.5) per spec
# ── TODO 4-C: ADMM hyper-parameters ─────────────────────────────────────────
TPV_RHO    = 1.0     # penalty (try 0.1 – 5.0)
TPV_ITER   = 50      # outer iterations (try 30 – 100)
TPV_CG     = 15      # CG inner iterations
# ──────────────────────────────────────────────────────────────────────────────

print(f"TpV-ADMM config: λ={TPV_LAM}, p={TPV_P}, ρ={TPV_RHO}, iter={TPV_ITER}")


In [ ]:
# ── Run TpV on the demo image ─────────────────────────────────────────────────
print("Running TpV-ADMM…  (this may take ~30 s on CPU, ~5 s on GPU)")
t0 = time.time()
with torch.no_grad():
    x_tpv = tpv_admm(y_demo, K, lam=TPV_LAM, p=TPV_P,
                     rho=TPV_RHO, n_iter=TPV_ITER, cg_iter=TPV_CG)
print(f"Done in {time.time()-t0:.1f} s")

psnr_tpv, ssim_tpv = compute_metrics(x_demo, x_tpv)
print(f"TpV  PSNR={psnr_tpv:.2f} dB   SSIM={ssim_tpv:.4f}  (noise_level={DEMO_NL})")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, t in zip(axes, [x_demo, y_demo, x_tpv],
                              ['Ground truth', f'Degraded δ={DEMO_NL}', f'TpV  PSNR={psnr_tpv:.2f}']):
    ax.imshow(img.detach().cpu().squeeze(), cmap='gray'); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ── TpV sensitivity study: vary p and λ ──────────────────────────────────────
# Useful for the oral discussion about parameter choice.
p_values   = [0.1, 0.2, 0.3, 0.4, 0.5]
lam_values = [0.005, 0.01, 0.02, 0.05]

print("p\\λ", end="")
for lam in lam_values:
    print(f"\t{lam:.3f}", end="")
print()

for p_val in p_values:
    print(f"{p_val}", end="")
    for lam_val in lam_values:
        with torch.no_grad():
            xr = tpv_admm(y_demo, K, lam=lam_val, p=p_val,
                          rho=TPV_RHO, n_iter=30, cg_iter=10)
        ps, _ = compute_metrics(x_demo, xr)
        print(f"\t{ps:.2f}", end="")
    print()
print("(PSNR dB — higher is better)")


## 5 · Method 2 — End-to-End: U-Net

### Architecture (recycled from course `residualLeraning2_6.py`)
We use the **Residual U-Net** with skip connections at each scale.
The key insight for **motion deblur** is that the residual shortcut helps the network learn
the *difference* between degraded and clean, rather than the clean image directly.

Architecture overview:
```
Input y (1×256×256)
  → Encoder: DoubleConv → ↓×2 → DoubleConv → ↓×2 → DoubleConv → ↓×2 → Bottleneck
  → Decoder: ↑×2 + skip → DoubleConv → ↑×2 + skip → ... → Conv 1×1
  → Output x̂ (1×256×256)
```

> **TODO 5-A** – Choose the base channel width `base_ch` (32 or 64 recommended).

> **TODO 5-B** – Choose number of epochs and learning rate.

> **TODO 5-C** – Optionally switch to `SimpleUNet` (no residual blocks) and compare.


In [ ]:
# ── Building blocks (identical to course residualLeraning2_6.py) ─────────────

class DoubleConv(nn.Module):
    """Two conv-BN-ReLU blocks used as the basic unit in the U-Net."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class ResidualDoubleConv(nn.Module):
    """DoubleConv with a residual shortcut — promotes gradient flow in deep nets."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1    = nn.Conv2d(in_ch,  out_ch, 3, padding=1)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.relu     = nn.ReLU(inplace=True)
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        identity = self.shortcut(x)
        h = self.relu(self.conv1(x))
        h = self.conv2(h)
        return self.relu(h + identity)


class DownBlock(nn.Module):
    """MaxPool ↓2 followed by a conv block."""
    def __init__(self, in_ch, out_ch, block_cls=DoubleConv):
        super().__init__()
        self.pool  = nn.MaxPool2d(2)
        self.block = block_cls(in_ch, out_ch)
    def forward(self, x): return self.block(self.pool(x))


class UpBlock(nn.Module):
    """Transposed-conv ↑2, concatenate skip, then conv block."""
    def __init__(self, in_ch, skip_ch, out_ch, block_cls=DoubleConv):
        super().__init__()
        self.up    = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = block_cls(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.block(torch.cat([skip, x], dim=1))


class ResidualUNet(nn.Module):
    """
    4-level Residual U-Net for image-to-image restoration.
    Input / output: (B, 1, 256, 256).
    """
    def __init__(self, in_ch=1, out_ch=1, base_ch=32):
        super().__init__()
        b = base_ch
        self.enc1       = ResidualDoubleConv(in_ch, b)
        self.enc2       = DownBlock(b,    2*b, ResidualDoubleConv)
        self.enc3       = DownBlock(2*b,  4*b, ResidualDoubleConv)
        self.bottleneck = DownBlock(4*b,  8*b, ResidualDoubleConv)
        self.dec3       = UpBlock(8*b, 4*b, 4*b, ResidualDoubleConv)
        self.dec2       = UpBlock(4*b, 2*b, 2*b, ResidualDoubleConv)
        self.dec1       = UpBlock(2*b,  b,   b,  ResidualDoubleConv)
        self.out_conv   = nn.Conv2d(b, out_ch, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        h  = self.bottleneck(s3)
        h  = self.dec3(h, s3)
        h  = self.dec2(h, s2)
        h  = self.dec1(h, s1)
        return self.out_conv(h)


# Quick shape sanity-check
_dummy = torch.zeros(2, 1, 256, 256)
_model = ResidualUNet(base_ch=32)
_out   = _model(_dummy)
n_params = sum(p.numel() for p in _model.parameters())
print(f"ResidualUNet output shape: {tuple(_out.shape)}")
print(f"Trainable parameters: {n_params:,}")


In [ ]:
# ── TODO 5-A / 5-B: Training hyper-parameters ────────────────────────────────
UNET_BASE_CH   = 32        # 32 or 64
UNET_EPOCHS    = 20        # number of training epochs
UNET_LR        = 1e-3      # Adam learning rate
UNET_NL        = 0.01      # noise level used during training
# ──────────────────────────────────────────────────────────────────────────────

torch.manual_seed(0)
unet_model = ResidualUNet(in_ch=1, out_ch=1, base_ch=UNET_BASE_CH).to(device)
unet_optimizer = torch.optim.Adam(unet_model.parameters(), lr=UNET_LR)
unet_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(unet_optimizer, T_max=UNET_EPOCHS)
unet_loss_fn   = nn.MSELoss()

UNET_WEIGHTS = WEIGHTS_DIR / 'UNet_FFHQ.pth'
unet_history  = {'train': [], 'val': []}

print(f"U-Net config: base_ch={UNET_BASE_CH}, epochs={UNET_EPOCHS}, lr={UNET_LR}")


In [ ]:
# ── Training loop (pattern from convolutionalNN2_5.py / residualLeraning2_6.py)
for epoch in range(UNET_EPOCHS):
    # ── Train ────────────────────────────────────────────────────────────────
    unet_model.train()
    epoch_loss = 0.0
    bar = tqdm(train_loader, desc=f'UNet epoch {epoch+1}/{UNET_EPOCHS}', leave=False)
    for step, x_batch in enumerate(bar, start=1):
        x_batch = x_batch.to(device)

        with torch.no_grad():                          # degrade on GPU without gradient
            y_batch  = K(x_batch)
            y_batch  = y_batch + gaussian_noise(y_batch, UNET_NL)

        x_pred = unet_model(y_batch)
        loss   = unet_loss_fn(x_pred, x_batch)

        unet_optimizer.zero_grad()
        loss.backward()
        unet_optimizer.step()

        epoch_loss += loss.item()
        bar.set_postfix(loss=f'{loss.item():.5f}', avg=f'{epoch_loss/step:.5f}')

    unet_history['train'].append(epoch_loss / len(train_loader))
    unet_scheduler.step()

    # ── Validation ────────────────────────────────────────────────────────────
    unet_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_val in val_loader:
            x_val  = x_val.to(device)
            y_val  = K(x_val) + gaussian_noise(K(x_val), UNET_NL)
            val_loss += unet_loss_fn(unet_model(y_val), x_val).item()
    unet_history['val'].append(val_loss / len(val_loader))

    print(f'Epoch {epoch+1:3d}  train={unet_history["train"][-1]:.6f}  val={unet_history["val"][-1]:.6f}')

torch.save(unet_model.state_dict(), UNET_WEIGHTS)
print(f'\nSaved U-Net weights → {UNET_WEIGHTS}')


In [ ]:
# ── Plot training curves & evaluate on demo image ─────────────────────────────
plt.figure(figsize=(7, 3))
plt.plot(unet_history['train'], label='Train loss')
plt.plot(unet_history['val'],   label='Val loss')
plt.title('ResidualUNet training loss'); plt.xlabel('Epoch'); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

unet_model.eval()
with torch.no_grad():
    x_unet = unet_model(y_demo).clamp(0, 1)

psnr_unet, ssim_unet = compute_metrics(x_demo, x_unet)
print(f"UNet PSNR={psnr_unet:.2f} dB  SSIM={ssim_unet:.4f}  (noise_level={DEMO_NL})")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, t in zip(axes, [x_demo, y_demo, x_unet],
                              ['Ground truth', f'Degraded δ={DEMO_NL}', f'U-Net  PSNR={psnr_unet:.2f}']):
    ax.imshow(img.detach().cpu().squeeze(), cmap='gray'); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()


## 6 · Method 3 — Generative: DDPM + DPS

### Theory

**DDPM (Denoising Diffusion Probabilistic Model)** learns a score function
$s_\theta(x_t, t) \approx \nabla_{x_t} \log p_t(x_t)$ by training a U-Net to predict
the noise $\epsilon$ added at each step of the forward process.

**DPS (Diffusion Posterior Sampling)** adapts unconditional DDPM to solve inverse problems
by adding a **data-consistency gradient** at each denoising step:

$$x_{t-1} \leftarrow \underbrace{\text{DDIM}(x_t, \hat{x}_0)}_\text{prior step} - \zeta_t \nabla_{x_t} \|K\hat{x}_0 - y^\delta\|^2$$

where $\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\epsilon_\theta}{\sqrt{\bar\alpha_t}}$
is the current best guess of the clean image.

### Procedure
1. **Train** a `TinyDiffusionUNet` on clean FFHQ images (noise prediction).
2. **Apply DPS** at test time to reconstruct from blurred+noisy observations.

> **TODO 6-A** – Set the DDPM hyper-parameters (T, β schedule, base_ch).

> **TODO 6-B** – Set the DPS guidance scale `zeta` and number of sampling steps.

> **TODO 6-C** – After training, comment on why DPS is slower than the feed-forward methods.


In [ ]:
# ── DDPM noise schedule ──────────────────────────────────────────────────────
# (identical to diffusionmodel2_9.py)
def make_beta_schedule(T, beta_start=1e-4, beta_end=2e-2):
    return torch.linspace(beta_start, beta_end, T)

def extract(coef, t, shape):
    """Index a 1-D coefficient tensor by timestep t and reshape for broadcasting."""
    out = coef.to(t.device)[t]
    return out.view(t.shape[0], *([1] * (len(shape) - 1)))

# ── TODO 6-A: DDPM hyper-parameters ─────────────────────────────────────────
NUM_DIFF_STEPS = 200
BETA_START     = 1e-4
BETA_END       = 2e-2
DIFF_BASE_CH   = 32       # channel width for TinyDiffusionUNet
# ──────────────────────────────────────────────────────────────────────────────

betas     = make_beta_schedule(NUM_DIFF_STEPS, BETA_START, BETA_END)
alphas    = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

print(f"DDPM schedule: T={NUM_DIFF_STEPS}, β ∈ [{betas[0]:.1e}, {betas[-1]:.1e}]")


In [ ]:
# ── TinyDiffusionUNet (from inversediffusionmodel2_10.py) ────────────────────
def sinusoidal_time_embedding(t, dim):
    """Sinusoidal positional encoding for diffusion timestep t."""
    half = dim // 2
    exp  = torch.arange(half, device=t.device, dtype=torch.float32) / half
    freq = torch.exp(-math.log(10000.0) * exp)
    ang  = t.float().unsqueeze(1) * freq.unsqueeze(0)
    emb  = torch.cat([torch.sin(ang), torch.cos(ang)], dim=1)
    if dim % 2 == 1:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
    return emb


class TimeConvBlock(nn.Module):
    """Conv block conditioned on the diffusion timestep via an additive projection."""
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1     = nn.Conv2d(in_ch,  out_ch, 3, padding=1)
        self.conv2     = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.act       = nn.ReLU(inplace=True)

    def forward(self, x, t_emb):
        h = self.act(self.conv1(x))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        return self.act(self.conv2(h))


class TinyDiffusionUNet(nn.Module):
    """
    Lightweight U-Net that takes a noisy image x_t and timestep t,
    and predicts the noise ε_t.
    Architecture matches inversediffusionmodel2_10.py.
    """
    def __init__(self, in_ch=1, base_ch=32, time_dim=128):
        super().__init__()
        b = base_ch
        self.time_dim  = time_dim
        self.time_mlp  = nn.Sequential(nn.Linear(time_dim, time_dim), nn.ReLU(), nn.Linear(time_dim, time_dim))
        self.enc1      = TimeConvBlock(in_ch, b,  time_dim)
        self.pool1     = nn.MaxPool2d(2)
        self.enc2      = TimeConvBlock(b,   2*b, time_dim)
        self.pool2     = nn.MaxPool2d(2)
        self.bottleneck= TimeConvBlock(2*b, 4*b, time_dim)
        self.up2       = nn.ConvTranspose2d(4*b, 2*b, 2, stride=2)
        self.dec2      = TimeConvBlock(4*b, 2*b, time_dim)
        self.up1       = nn.ConvTranspose2d(2*b,  b,  2, stride=2)
        self.dec1      = TimeConvBlock(2*b,  b,  time_dim)
        self.out_conv  = nn.Conv2d(b, in_ch, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(sinusoidal_time_embedding(t, self.time_dim))
        s1    = self.enc1(x, t_emb)
        s2    = self.enc2(self.pool1(s1), t_emb)
        h     = self.bottleneck(self.pool2(s2), t_emb)
        h     = self.dec2(torch.cat([self.up2(h), s2], 1), t_emb)
        h     = self.dec1(torch.cat([self.up1(h), s1], 1), t_emb)
        return self.out_conv(h)


diff_model = TinyDiffusionUNet(in_ch=1, base_ch=DIFF_BASE_CH, time_dim=128)
n_p = sum(p.numel() for p in diff_model.parameters())
print(f"TinyDiffusionUNet — {n_p:,} parameters")


In [ ]:
# ── DDPM training loop (pattern from diffusionmodel2_9.py) ───────────────────
DIFF_EPOCHS  = 30          # increase for better quality
DIFF_LR      = 2e-4
DIFF_WEIGHTS = WEIGHTS_DIR / 'DDPM_FFHQ.pth'

torch.manual_seed(0)
diff_model     = TinyDiffusionUNet(in_ch=1, base_ch=DIFF_BASE_CH).to(device)
diff_optimizer = torch.optim.AdamW(diff_model.parameters(), lr=DIFF_LR, weight_decay=1e-4)
diff_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(diff_optimizer, T_max=DIFF_EPOCHS)
diff_history   = []

for epoch in range(DIFF_EPOCHS):
    diff_model.train()
    ep_loss = 0.0
    bar = tqdm(train_loader, desc=f'DDPM epoch {epoch+1}/{DIFF_EPOCHS}', leave=False)
    for step, x0 in enumerate(bar, start=1):
        x0 = x0.to(device)
        # Sample random timesteps and noise
        t     = torch.randint(0, NUM_DIFF_STEPS, (x0.shape[0],), device=device)
        noise = torch.randn_like(x0)
        # Forward diffusion: q(x_t | x_0)
        x_t   = extract(alpha_bars.sqrt(), t, x0.shape) * x0 +                 extract((1 - alpha_bars).sqrt(), t, x0.shape) * noise
        # Predict noise and compute MSE loss
        loss  = F.mse_loss(diff_model(x_t, t), noise)

        diff_optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(diff_model.parameters(), 1.0)
        diff_optimizer.step()

        ep_loss += loss.item()
        bar.set_postfix(loss=f'{loss.item():.5f}')

    diff_history.append(ep_loss / len(train_loader))
    diff_scheduler.step()
    print(f'DDPM epoch {epoch+1:3d}  loss={diff_history[-1]:.6f}')

torch.save(diff_model.state_dict(), DIFF_WEIGHTS)
print(f'Saved DDPM weights → {DIFF_WEIGHTS}')

plt.figure(figsize=(6, 3))
plt.plot(diff_history); plt.title('DDPM training loss'); plt.xlabel('Epoch'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── DPS reconstruction (from inversediffusionmodel2_10.py) ───────────────────
def predict_x0(x_t, eps_pred, t, alpha_bars):
    """
    Tweedie formula: reconstruct x_0 from x_t and the predicted noise ε.
    x̂_0 = (x_t - √(1-ᾱ_t) · ε) / √ᾱ_t
    """
    return (x_t - extract((1 - alpha_bars).sqrt(), t, x_t.shape) * eps_pred) /            extract(alpha_bars.sqrt(), t, x_t.shape)


def ddim_step(x_t, x0_hat, eps_pred, t_next, alpha_bars):
    """Deterministic DDIM reverse step (η=0)."""
    if t_next < 0:
        return x0_hat
    ab_next = alpha_bars[t_next].to(x_t.device)
    return torch.sqrt(ab_next) * x0_hat + torch.sqrt(1 - ab_next) * eps_pred


def dps_reconstruct(model, y_delta, K, alpha_bars,
                    sigma_y=0.01, sample_steps=40, guidance_scale=0.15):
    """
    Diffusion Posterior Sampling for linear inverse problems.

    At each DDIM step we:
      1. Predict ε_θ(x_t, t)  (score function)
      2. Compute x̂_0 via Tweedie
      3. Compute data loss  L = ‖K x̂_0 - y^δ‖² / (2σ_y²)
      4. Take gradient of L w.r.t. x_t  (guidance)
      5. Update x_{t-1} = DDIM(x_t) - ζ · ∇L

    Parameters
    ----------
    model         : trained TinyDiffusionUNet
    y_delta       : degraded observation (1,1,H,W)
    K             : MotionBlurOperator
    sigma_y       : observation noise std (set = noise_level used during degradation)
    sample_steps  : number of DDIM reverse steps (fewer → faster but lower quality)
    guidance_scale: step size ζ for the data consistency gradient
    """
    schedule = torch.linspace(NUM_DIFF_STEPS - 1, 0, sample_steps, dtype=torch.long, device=device)
    x = torch.randn_like(y_delta)

    for i in range(len(schedule) - 1):
        t_curr = int(schedule[i].item())
        t_next = int(schedule[i + 1].item())
        t      = torch.full((x.shape[0],), t_curr, device=device, dtype=torch.long)

        x          = x.detach().requires_grad_(True)
        eps_pred   = model(x, t)
        x0_hat     = predict_x0(x, eps_pred, t, alpha_bars).clamp(0.0, 1.0)

        # Data-consistency gradient (DPS guidance)
        data_loss  = torch.mean((K(x0_hat) - y_delta) ** 2) / (2 * sigma_y ** 2)
        grad       = torch.autograd.grad(data_loss, x)[0]

        with torch.no_grad():
            x_next = ddim_step(x, x0_hat, eps_pred, t_next, alpha_bars)
            x      = (x_next - guidance_scale * grad).clamp(0.0, 1.0)

    # Final denoising step
    with torch.no_grad():
        t0       = torch.zeros((x.shape[0],), device=device, dtype=torch.long)
        eps_pred = model(x, t0)
        return predict_x0(x, eps_pred, t0, alpha_bars).clamp(0.0, 1.0)


# ── TODO 6-B: DPS hyper-parameters ──────────────────────────────────────────
DPS_STEPS    = 40           # DDIM steps (50–200 for final eval)
DPS_GUIDANCE = 0.15         # guidance scale ζ (try 0.05 – 0.5)
DPS_SIGMA    = DEMO_NL      # should match the noise level of the observation
# ──────────────────────────────────────────────────────────────────────────────

print(f"DPS config: steps={DPS_STEPS}, ζ={DPS_GUIDANCE}, σ_y={DPS_SIGMA}")


In [ ]:
# ── Load best DDPM weights and run DPS on demo image ─────────────────────────
diff_model.load_state_dict(torch.load(DIFF_WEIGHTS, map_location=device, weights_only=True))
diff_model.eval()

print(f"Running DPS ({DPS_STEPS} steps) …  (expect ~1-3 min on GPU)")
t0 = time.time()
x_dps = dps_reconstruct(diff_model, y_demo, K, alpha_bars,
                        sigma_y=DPS_SIGMA, sample_steps=DPS_STEPS, guidance_scale=DPS_GUIDANCE)
print(f"Done in {time.time()-t0:.1f} s")

psnr_dps, ssim_dps = compute_metrics(x_demo, x_dps)
print(f"DPS  PSNR={psnr_dps:.2f} dB   SSIM={ssim_dps:.4f}  (noise_level={DEMO_NL})")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, t in zip(axes, [x_demo, y_demo, x_dps],
                              ['Ground truth', f'Degraded δ={DEMO_NL}', f'DPS  PSNR={psnr_dps:.2f}']):
    ax.imshow(img.detach().cpu().squeeze(), cmap='gray'); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()


## 7 · Method 4 — Hybrid: Plug-and-Play HQS (PnP-HQS)

### Theory

**Half Quadratic Splitting (HQS)** rewrites the regularised problem:

$$\min_x \frac{1}{2}\|Kx - y\|^2 + \lambda R(x)$$

by introducing an auxiliary variable $z = x$ and augmented-Lagrangian penalty $\mu$:

$$\min_{x,z} \frac{1}{2}\|Kx - y\|^2 + \frac{\mu}{2}\|x - z\|^2 + \lambda R(z)$$

This splits into two simple sub-problems:

1. **x-subproblem** (data-fidelity + quadratic): solved in **frequency domain** (Fourier trick):

$$x^{k+1} = \mathcal{F}^{-1}\!\left(\frac{\bar H \cdot \mathcal{F}(y) + \mu\,\mathcal{F}(z^k)}{|H|^2 + \mu}\right)$$

2. **z-subproblem** (proximal of R): in **Plug-and-Play** we replace the proximal operator with
   a **pretrained denoiser** $D_\sigma$ (e.g. the trained U-Net):

$$z^{k+1} = D_\sigma\!\left(x^{k+1}\right)$$

where $\sigma = \sqrt{\lambda / \mu}$ is the effective noise level passed to the denoiser.

> **TODO 7-A** – Choose `mu` (HQS penalty) and `n_iter`.

> **TODO 7-B** – Choose which pretrained denoiser to plug in (`unet_model` or a fresh one).

> **TODO 7-C** – Comment in the report: why does PnP-HQS benefit from the Fourier trick for motion blur?


In [ ]:
# ── Fourier-domain x-subproblem solver ───────────────────────────────────────
def kernel_to_otf(kernel: torch.Tensor, img_shape) -> torch.Tensor:
    """
    Convert a spatial PSF kernel to its Optical Transfer Function (OTF)
    via zero-padding and FFT2.  Needed for the efficient Fourier x-update.

    Returns the complex OTF of shape img_shape (H, W).
    """
    h, w = img_shape
    psf  = torch.zeros(h, w, device=kernel.device)
    k    = kernel.squeeze()
    kh, kw = k.shape
    psf[:kh, :kw] = k
    # Circular shift so PSF is centred at (0,0) for correct FFT convolution
    psf = torch.roll(psf, shifts=(-kh // 2, -kw // 2), dims=(0, 1))
    return torch.fft.fft2(psf)   # complex tensor (H, W)


def x_update_fourier(y: torch.Tensor, z: torch.Tensor,
                     H_fft: torch.Tensor, mu: float) -> torch.Tensor:
    """
    Solve the x-subproblem in the Fourier domain:
        x* = F^{-1}( (H* Y + μ Z) / (|H|^2 + μ) )

    Parameters
    ----------
    y     : observation (1, 1, H, W)
    z     : current auxiliary variable (1, 1, H, W)
    H_fft : OTF of K, complex tensor (H, W)
    mu    : HQS penalty scalar

    Returns
    -------
    x_new : (1, 1, H, W) updated primal variable
    """
    Y     = torch.fft.fft2(y.squeeze(1))       # (1, H, W) complex
    Z     = torch.fft.fft2(z.squeeze(1))
    H     = H_fft.unsqueeze(0)                 # (1, H, W)
    numer = H.conj() * Y + mu * Z
    denom = H.abs() ** 2 + mu
    X     = numer / denom
    x_new = torch.fft.ifft2(X).real.unsqueeze(1)
    return x_new.clamp(0.0, 1.0)


# Pre-compute OTF from the motion-blur kernel
H_fft = kernel_to_otf(K.kernel.squeeze(), img_shape=(256, 256))
print(f"OTF computed: shape={H_fft.shape}, dtype={H_fft.dtype}")


In [ ]:
# ── PnP-HQS main loop ─────────────────────────────────────────────────────────
def pnp_hqs(y: torch.Tensor,
            K,
            H_fft: torch.Tensor,
            denoiser: nn.Module,
            mu: float,
            n_iter: int,
            lam: float = 0.01) -> torch.Tensor:
    """
    Plug-and-Play HQS for blind-free linear inverse problem.

    Algorithm
    ---------
    Initialise z = y
    For k = 1 … n_iter:
        x ← Fourier_solve(y, z, H, μ)          # data fidelity
        z ← Denoiser(x, σ=√(λ/μ))              # learned prior
    Return x

    Parameters
    ----------
    y        : degraded observation (1, 1, H, W)
    K        : MotionBlurOperator (used only for fallback; x-step uses H_fft)
    H_fft    : complex OTF tensor (H, W) from kernel_to_otf()
    denoiser : pretrained nn.Module, callable as denoiser(x) → x̂
    mu       : HQS quadratic penalty (controls trade-off between data fidelity & prior)
    n_iter   : number of alternating iterations
    lam      : regularisation weight (σ_denoiser = sqrt(lam/mu))

    Returns
    -------
    x : reconstructed image (1, 1, H, W), values in [0, 1]
    """
    z = y.clone()

    for k in range(n_iter):
        # x-update: analytical frequency-domain solve
        x = x_update_fourier(y, z, H_fft, mu)

        # z-update: plug in the learned denoiser (replace proximal operator)
        with torch.no_grad():
            z = denoiser(x).clamp(0.0, 1.0)

    return x.clamp(0.0, 1.0)


# ── TODO 7-A: PnP-HQS hyper-parameters ──────────────────────────────────────
PNP_MU    = 0.05     # quadratic penalty μ  (try 0.01 – 1.0)
PNP_ITER  = 20       # number of outer iterations  (try 10 – 50)
PNP_LAM   = 0.01     # regularisation strength
# ── TODO 7-B: Choose denoiser ─────────────────────────────────────────────────
# Use the already-trained U-Net as the plug-and-play denoiser.
# Make sure unet_model is already trained (Section 5) before running this cell.
pnp_denoiser = unet_model    # or swap in any other nn.Module
# ──────────────────────────────────────────────────────────────────────────────

print(f"PnP-HQS config: μ={PNP_MU}, iter={PNP_ITER}, denoiser={type(pnp_denoiser).__name__}")


In [ ]:
# ── Run PnP-HQS on the demo image ────────────────────────────────────────────
pnp_denoiser.eval()

print("Running PnP-HQS…")
t0 = time.time()
with torch.no_grad():
    x_pnp = pnp_hqs(y_demo, K, H_fft, pnp_denoiser,
                    mu=PNP_MU, n_iter=PNP_ITER, lam=PNP_LAM)
print(f"Done in {time.time()-t0:.2f} s")

psnr_pnp, ssim_pnp = compute_metrics(x_demo, x_pnp)
print(f"PnP-HQS  PSNR={psnr_pnp:.2f} dB   SSIM={ssim_pnp:.4f}  (noise_level={DEMO_NL})")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, t in zip(axes, [x_demo, y_demo, x_pnp],
                              ['Ground truth', f'Degraded δ={DEMO_NL}', f'PnP-HQS  PSNR={psnr_pnp:.2f}']):
    ax.imshow(img.detach().cpu().squeeze(), cmap='gray'); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ── PnP-HQS sensitivity: vary μ ───────────────────────────────────────────────
mu_values = [0.01, 0.05, 0.1, 0.5, 1.0]
print("μ\t\tPSNR\tSSIM")
for mu_val in mu_values:
    with torch.no_grad():
        xr = pnp_hqs(y_demo, K, H_fft, pnp_denoiser,
                     mu=mu_val, n_iter=PNP_ITER, lam=PNP_LAM)
    ps, ss = compute_metrics(x_demo, xr)
    print(f"μ={mu_val:.3f}\t\t{ps:.2f}\t{ss:.4f}")


## 8 · Final Evaluation & Comparison

This section evaluates **all four methods** on the full test set across **all four noise levels**.
Results are reported as mean PSNR / SSIM ± std (as required by the project spec).


In [ ]:
# ── Evaluation loop — all methods × all noise levels ─────────────────────────
# NOTE: Make sure all four method weights are loaded / models are trained before
#       running this cell.  DPS is slow; consider reducing N_TEST for quick runs.

N_EVAL = 50       # number of test images to average over (use full N_TEST for final report)

results = {nl: {name: {'psnr': [], 'ssim': []} for name in ['TpV', 'UNet', 'DPS', 'PnP-HQS']}
           for nl in NOISE_LEVELS}

unet_model.eval()
diff_model.eval()

for nl in NOISE_LEVELS:
    print(f"\n=== Noise level δ = {nl} ===")
    for idx in tqdm(range(min(N_EVAL, len(test_dataset))), desc=f'Eval δ={nl}'):
        x_true = test_dataset[idx].unsqueeze(0).to(device)

        with torch.no_grad():
            y_clean = K(x_true)
            y_noisy = y_clean + gaussian_noise(y_clean, nl)

        # Method 1: TpV
        with torch.no_grad():
            xr_tpv = tpv_admm(y_noisy, K, lam=TPV_LAM, p=TPV_P,
                               rho=TPV_RHO, n_iter=TPV_ITER, cg_iter=TPV_CG)
        ps, ss = compute_metrics(x_true, xr_tpv)
        results[nl]['TpV']['psnr'].append(ps); results[nl]['TpV']['ssim'].append(ss)

        # Method 2: UNet
        with torch.no_grad():
            xr_unet = unet_model(y_noisy).clamp(0, 1)
        ps, ss = compute_metrics(x_true, xr_unet)
        results[nl]['UNet']['psnr'].append(ps); results[nl]['UNet']['ssim'].append(ss)

        # Method 3: DPS  (slow — reduce DPS_STEPS for speed)
        xr_dps = dps_reconstruct(diff_model, y_noisy, K, alpha_bars,
                                 sigma_y=nl, sample_steps=DPS_STEPS, guidance_scale=DPS_GUIDANCE)
        ps, ss = compute_metrics(x_true, xr_dps)
        results[nl]['DPS']['psnr'].append(ps); results[nl]['DPS']['ssim'].append(ss)

        # Method 4: PnP-HQS
        with torch.no_grad():
            xr_pnp = pnp_hqs(y_noisy, K, H_fft, pnp_denoiser,
                              mu=PNP_MU, n_iter=PNP_ITER, lam=PNP_LAM)
        ps, ss = compute_metrics(x_true, xr_pnp)
        results[nl]['PnP-HQS']['psnr'].append(ps); results[nl]['PnP-HQS']['ssim'].append(ss)

print("\nEvaluation done.")


In [ ]:
# ── Print summary table ───────────────────────────────────────────────────────
print(f"{'Method':<12} {'δ':>6}  {'PSNR (dB)':>12}  {'SSIM':>10}")
print("-" * 48)
for nl in NOISE_LEVELS:
    for method in ['TpV', 'UNet', 'DPS', 'PnP-HQS']:
        ps = results[nl][method]['psnr']
        ss = results[nl][method]['ssim']
        print(f"{method:<12} {nl:>6.3f}  {np.mean(ps):>6.2f} ± {np.std(ps):.2f}  "
              f"{np.mean(ss):>6.4f} ± {np.std(ss):.4f}")
    print()


In [ ]:
# ── Comparison bar-charts (PSNR) ─────────────────────────────────────────────
methods = ['TpV', 'UNet', 'DPS', 'PnP-HQS']
x_pos   = np.arange(len(NOISE_LEVELS))
width   = 0.18

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, method in enumerate(methods):
    psnr_means = [np.mean(results[nl][method]['psnr']) for nl in NOISE_LEVELS]
    ssim_means = [np.mean(results[nl][method]['ssim']) for nl in NOISE_LEVELS]
    axes[0].bar(x_pos + i * width, psnr_means, width, label=method)
    axes[1].bar(x_pos + i * width, ssim_means, width, label=method)

axes[0].set_title('PSNR comparison'); axes[0].set_ylabel('PSNR (dB)')
axes[1].set_title('SSIM comparison'); axes[1].set_ylabel('SSIM')
for ax in axes:
    ax.set_xticks(x_pos + width * 1.5)
    ax.set_xticklabels([f'δ={nl}' for nl in NOISE_LEVELS])
    ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Method comparison — FFHQ Motion Deblur & Denoising')
plt.tight_layout(); plt.show()


In [ ]:
# ── Visual comparison grid (one image per noise level) ────────────────────────
fig, axes = plt.subplots(len(NOISE_LEVELS), 6,
                         figsize=(18, 3.5 * len(NOISE_LEVELS)))

x_vis = test_dataset[0].unsqueeze(0).to(device)
for row, nl in enumerate(NOISE_LEVELS):
    with torch.no_grad():
        y_v   = K(x_vis) + gaussian_noise(K(x_vis), nl)
        xr_t  = tpv_admm(y_v, K, lam=TPV_LAM, p=TPV_P, rho=TPV_RHO, n_iter=TPV_ITER, cg_iter=TPV_CG)
        xr_u  = unet_model(y_v).clamp(0, 1)
        xr_p  = pnp_hqs(y_v, K, H_fft, pnp_denoiser, mu=PNP_MU, n_iter=PNP_ITER, lam=PNP_LAM)
    xr_d  = dps_reconstruct(diff_model, y_v, K, alpha_bars, sigma_y=nl,
                             sample_steps=DPS_STEPS, guidance_scale=DPS_GUIDANCE)

    imgs   = [x_vis, y_v, xr_t, xr_u, xr_d, xr_p]
    titles = ['GT', f'Degraded δ={nl}', 'TpV', 'UNet', 'DPS', 'PnP-HQS']
    for ax, img, t in zip(axes[row], imgs, titles):
        ax.imshow(img.detach().cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
        ax.set_title(t, fontsize=9); ax.axis('off')

plt.suptitle('Visual comparison across noise levels', fontsize=13)
plt.tight_layout(); plt.show()


## 9 · Discussion Checklist

Use this section to record your analysis for the oral presentation.

**Parameters chosen and rationale**
- Motion angle: `MOTION_ANGLE = …`  → justify
- TpV: `p = …`, `λ = …`, `ρ = …`  → explain how you searched this space (Section 4 sensitivity table)
- UNet: `base_ch = …`, `epochs = …` → relate to dataset size and GPU time
- DPS: `sample_steps = …`, `ζ = …`, `σ_y = …` → explain guidance scale trade-off
- PnP-HQS: `μ = …`, `n_iter = …`  → explain μ role (Section 7 sensitivity table)

**Expected observations**
1. TpV is the slowest per image (per-image optimisation) but requires no training data.
2. U-Net is fastest at inference but needs a large training set; may struggle at high noise.
3. DPS is the slowest overall (many UNet forward passes) but leverages a generative prior.
4. PnP-HQS balances speed and quality; benefits from the Fourier closed-form x-update.

> **TODO 9-A** – Fill in the actual numbers from your runs.

> **TODO 9-B** – Add qualitative observations: which method preserves fine face details best?

> **TODO 9-C** – Discuss failure modes at `δ = 0.1` for each method.


In [ ]:
# ── Save all results to disk for the report ──────────────────────────────────
import json, pickle

summary = {}
for nl in NOISE_LEVELS:
    summary[str(nl)] = {}
    for method in ['TpV', 'UNet', 'DPS', 'PnP-HQS']:
        ps = results[nl][method]['psnr']
        ss = results[nl][method]['ssim']
        summary[str(nl)][method] = {
            'psnr_mean': float(np.mean(ps)), 'psnr_std': float(np.std(ps)),
            'ssim_mean': float(np.mean(ss)), 'ssim_std': float(np.std(ss)),
        }

with open('/content/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Results saved to /content/results_summary.json")
print(json.dumps(summary, indent=2))
